# Report for practical lesson 02: Convolutional Neural Network

In [5]:
import pandas as pd
import os, sys
import numpy as np
project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root_path not in sys.path:
    sys.path.append(project_root_path)

from sklearn.model_selection import train_test_split
import torch

import matplotlib.pyplot as plt
import torchvision.transforms as transforms

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter # Sử dụng TensorBoard logger
from sklearn.metrics import classification_report
import numpy as np
import os
import time

This box imports customized modules

In [6]:
from src.data.load import load_data
from src.train_model import train_model

Crucial variables

In [7]:
DATA_PATH = r"c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\lesson-02\dataset"

## Data Preprocessing

### Work 01: Load dataset

In [8]:
original_train, original_val, original_test, classes = load_data(
    data_root=DATA_PATH,
    num_workers=0, pin_memory=False,
    batch_size=32
)

### Work 02: Turn dataset into tensor format

In [9]:
def create_tensor_loaders(train_loader, val_loader, test_loader):
    """
    Create tensor transforms and corresponding DataLoaders for train, validation, and test datasets.

    Args:
        train_loader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        val_loader (torch.utils.data.DataLoader): DataLoader for the validation dataset.
        test_loader (torch.utils.data.DataLoader): DataLoader for the test dataset.

    Returns:
        tuple: (train_tensor_loader, val_tensor_loader, test_tensor_loader)
            - train_tensor_loader: DataLoader with tensor transform applied to training data.
            - val_tensor_loader: DataLoader with tensor transform applied to validation data.
            - test_tensor_loader: DataLoader with tensor transform applied to test data.
    """

    train_tensor_loader = torch.utils.data.DataLoader(
        train_loader.dataset,
        batch_size=train_loader.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )

    val_tensor_loader = torch.utils.data.DataLoader(
        val_loader.dataset,
        batch_size=val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    test_tensor_loader = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    return train_tensor_loader, val_tensor_loader, test_tensor_loader

In [10]:
train_tensor_loader, val_tensor_loader, test_tensor_loader = create_tensor_loaders(
    original_train, original_val, original_test
)

images, labels = next(iter(train_tensor_loader))
print("Tensor shape:", images.shape)
print("Data type:", images.dtype)

train_tensor_loader

Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


### Work 02: Transform data into grayscale

In [11]:
def to_grayscale(train_loader, val_loader, test_loader):
    """
    Convert RGB images to grayscale for each data loader
    
    Args:
        train_loader: DataLoader with RGB images
        val_loader: DataLoader with RGB images
        test_loader: DataLoader with RGB images
        
    Returns:
        Three DataLoaders containing grayscale images
    """
    grayscale_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1)
    ])
    
    # Create new dataloaders with grayscale transform
    train_gray = torch.utils.data.DataLoader(
        train_loader.dataset,
        batch_size=train_loader.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )
    
    val_gray = torch.utils.data.DataLoader(
        val_loader.dataset,
        batch_size=val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    test_gray = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    return train_gray, val_gray, test_gray

In [12]:
train, val, test = to_grayscale(
    train_tensor_loader, 
    val_tensor_loader, 
    test_tensor_loader
)

In [13]:
images, labels = next(iter(train))
print("Tensor shape:", images.shape)
print("Data type:", images.dtype)

train_tensor_loader

Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


## Session 01

### Worrk 01: Build Neural Network

The current model was establish via Python code (`lesson-02\networks\model01.py`). The bellow code snippet is to load model into this notebook.

In [14]:
from src.networks.model01 import model01

In [15]:
print("Number of classes:", len(classes))
instance01 = model01(num_classes=len(classes))
print("Instance 01 created successfully.")

Number of classes: 21
Instance 01 created successfully.


To print model description, run this.

In [16]:
print(instance01)

model01(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=21, bias=True)
)


### Work 02: Train model

In [17]:
from src.train_model import train_model

In [18]:
def preprocessing_fn(batch: torch.Tensor) -> torch.Tensor:
    # Convert from [B, 3, 224, 224] -> [B, 1, 28, 28]
    gray = batch.mean(dim=1, keepdim=True)  # RGB to grayscale
    resized = torch.nn.functional.interpolate(gray, size=(28, 28), mode='bilinear', align_corners=False)
    return resized

In [19]:
dev = print('cuda' if torch.cuda.is_available() else 'cpu')

cuda


In [43]:
trained_model, history, best_ckpt_path, test_metrics = train_model(
    train_loader=train,
    val_loader=val,
    model=instance01,  
    epochs=10,
    lr=0.005,
    preprocessing_fn=preprocessing_fn,
    device=dev,
    model_args=None 
)

Epoch 1/10 - train:  71%|███████   | 200/283 [01:55<00:55,  1.51batch/s, loss=2.93]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/10 - val: 100%|██████████| 32/32 [00:16<00:00,  1.97batch/s, val_loss=2.89]


Epoch 1: train_loss=2.9209 val_loss=2.8913 acc=0.1056 prec=0.0245 rec=0.0667 f1=0.0322


Epoch 2/10 - train:  68%|██████▊   | 192/283 [01:47<00:56,  1.62batch/s, loss=2.86]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2/10 - val: 100%|██████████| 32/32 [00:17<00:00,  1.86batch/s, val_loss=2.77]


Epoch 2: train_loss=2.8458 val_loss=2.7744 acc=0.1633 prec=0.0786 rec=0.1147 f1=0.0706


Epoch 3/10 - train:  88%|████████▊ | 249/283 [02:23<00:21,  1.56batch/s, loss=2.8] c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 3/10 - val: 100%|██████████| 32/32 [00:15<00:00,  2.01batch/s, val_loss=2.8] 


...

Epoch 8: train_loss=2.7383 val_loss=2.6859 acc=0.1753 prec=0.1042 rec=0.1329 f1=0.0909


Epoch 9/10 - train:   0%|          | 1/283 [00:01<06:44,  1.43s/batch, loss=2.47]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 9/10 - val: 100%|██████████| 32/32 [00:20<00:00,  1.59batch/s, val_loss=2.71]


Epoch 9: train_loss=2.7135 val_loss=2.7121 acc=0.1713 prec=0.1005 rec=0.1314 f1=0.0956


Epoch 10/10 - train:  76%|███████▌  | 214/283 [02:06<00:42,  1.63batch/s, loss=2.72]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 10/10 - val: 100%|██████████| 32/32 [00:20<00:00,  1.53batch/s, val_loss=2.7] 

Epoch 10: train_loss=2.7192 val_loss=2.7036 acc=0.1653 prec=0.0838 rec=0.1279 f1=0.0907


In [ ]:
import pandas as pd

## Session 02

### Work 01: Neural Network Design

In [21]:
from src.networks.model02 import GoogLeNet

In [23]:
instance02 = GoogLeNet(num_classes=len(classes))
print(instance02)

GoogLeNet(
  (stem): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (2): ConvBlock(
      (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (3): ConvBlock(
      (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (4): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (inception_3a): Inception(
    (branch1): ConvBlock(
      (conv): Conv2d(192, 

### Work 02: Train model

In [27]:
device_for_instance02 = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device_for_instance02}')

Using device: cuda


In [42]:
def train_instance02(
    # --- Nhóm Cốt lõi ---
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    
    # --- Nhóm Cấu hình ---
    num_epochs: int,
    device: torch.device,
    
    # --- Nhóm MLOps ---
    checkpoint_dir: str,
    logger: SummaryWriter
):
    """
    Hàm train chuyên biệt cho GoogLeNet (InceptionV1).
    Xử lý 3 output khi training và 1 output khi validation.
    """
    
    # Tạo thư mục checkpoint nếu chưa có
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
        
    best_val_loss = float('inf')
    
    print(f"Bắt đầu training trên thiết bị: {device}")
    
    for epoch in range(num_epochs):
        
        # ==========================
        #      PHA TRAINING
        # ==========================
        model.train() # QUAN TRỌNG: Bật chế độ train
                      # (GoogLeNet sẽ trả về 3 output)
        
        running_train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Xóa gradient
            optimizer.zero_grad()
            
            # --- PHẦN LOGIC ĐẶC BIỆT CỦA GOOGLENET ---
            # Forward pass - nhận 3 output
            main_output, aux1_output, aux2_output = model(inputs)
            
            # Tính loss cho cả 3
            loss_main = criterion(main_output, labels)
            loss_aux1 = criterion(aux1_output, labels)
            loss_aux2 = criterion(aux2_output, labels)
            
            # Tổng loss theo trọng số của paper
            loss = loss_main + 0.3 * loss_aux1 + 0.3 * loss_aux2
            # --- KẾT THÚC PHẦN LOGIC ĐẶC BIỆT ---
            
            # Backward
            loss.backward()
            
            # Optimize
            optimizer.step()
            
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        
        # ==========================
        #     PHA VALIDATION
        # ==========================
        model.eval() # QUAN TRỌNG: Bật chế độ eval
                     # (GoogLeNet sẽ chỉ trả về 1 output chính)
        
        running_val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad(): # Không cần tính gradient
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # Forward pass - chỉ nhận 1 output
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                # Lấy dự đoán để tính classification report
                _, preds = torch.max(outputs, 1)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        avg_val_loss = running_val_loss / len(val_loader)
        
        # Nối các kết quả từ các batch
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        
        # ==========================
        #     LOGGING & CHECKPOINT
        # ==========================
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {avg_val_loss:.4f}")
        
        # Log ra TensorBoard
        logger.add_scalar('Loss/train', avg_train_loss, epoch)
        logger.add_scalar('Loss/validation', avg_val_loss, epoch)
        
        # In classification report (precision, recall, f1)
        # Lấy tên class từ loader (nếu có) hoặc tạo tên giả
        try:
            class_names = val_loader.dataset.classes
        except:
            class_names = [f'Class {i}' for i in range(len(np.unique(all_labels)))]
            
        report = classification_report(
            all_labels, 
            all_preds, 
            target_names=class_names, 
            zero_division=0,
            digits=4
        )
        print("--- Báo cáo đánh giá (Precision, Recall, F1) ---")
        print(report)
        
        # Log F1-score (macro avg) vào TensorBoard
        # (Bạn có thể parse 'report' hoặc tính riêng, ở đây tôi log F1-score từ report)
        report_dict = classification_report(all_labels, all_preds, zero_division=0, output_dict=True)
        logger.add_scalar('F1-Score/macro_avg', report_dict['macro avg']['f1-score'], epoch)
        logger.add_scalar('Accuracy/validation', report_dict['accuracy'], epoch)

        # Lưu checkpoint của model tốt nhất
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save(model.state_dict(), save_path)
            print(f"==> Model tốt nhất đã được lưu tại: {save_path}")
            
    print("\nTraining hoàn tất.")
    logger.close()

In [35]:
for images, labels in train:
    print("Number of batches:", len(train))
    print("Number of classes:", len(classes))
    print("Tensor shape:", images.shape)
    print("Data type:", images.dtype)
    break

Number of batches: 283
Number of classes: 21
Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


In [37]:
NUM_CLASSES = 21
NUM_EPOCHS = 15
LEARNING_RATE = 0.001
MODEL_NAME = "GoogLeNet"
CHECKPOINT_DIR = f"./checkpoints/{MODEL_NAME}"
LOG_DIR = f"./logs/{MODEL_NAME}_{int(time.time())}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
print(f"Đang khởi tạo các thành phần cho {MODEL_NAME}...")
model = GoogLeNet(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
logger = SummaryWriter(log_dir=LOG_DIR)

print(f"Model: {MODEL_NAME}, Num Classes: {NUM_CLASSES}")
print(f"Device: {device}")
print(f"Checkpoints sẽ lưu tại: {CHECKPOINT_DIR}")
print(f"Logs TensorBoard sẽ lưu tại: {LOG_DIR}")

Đang khởi tạo các thành phần cho GoogLeNet...
Model: GoogLeNet, Num Classes: 21
Device: cuda
Checkpoints sẽ lưu tại: ./checkpoints/GoogLeNet
Logs TensorBoard sẽ lưu tại: ./logs/GoogLeNet_1762075265


In [41]:
train_instance02(
    model=model,
    train_loader=train, # <-- Biến 'train' của bạn
    val_loader=val,     # <-- Biến 'val' của bạn
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger
)

Bắt đầu training trên thiết bị: cuda


c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 1/15
  Train Loss: 4.3777
  Val Loss:   3.1371
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000        41
     Class 1     0.1500    0.3956    0.2175        91
     Class 2     0.0000    0.0000    0.0000        30
     Class 3     0.1761    0.5435    0.2660        46
     Class 4     0.0000    0.0000    0.0000        47
     Class 5     0.0000    0.0000    0.0000        21
     Class 6     0.1429    0.0235    0.0404        85
     Class 7     0.0000    0.0000    0.0000        28
     Class 8     0.0000    0.0000    0.0000        47
     Class 9     0.2500    0.0580    0.0941        69
    Class 10     0.0250    0.0233    0.0241        43
    Class 11     0.0476    0.0385    0.0426        26
    Class 12     0.4643    0.3514    0.4000        37
    Class 13     0.1818    0.7143    0.2899        28
    Class 14     0.1382    0.5758    0.2229        66
    Class 15     0.0000    0.000

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 2/15
  Train Loss: 4.0647
  Val Loss:   2.3989
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000        41
     Class 1     0.4032    0.2747    0.3268        91
     Class 2     0.1034    0.3000    0.1538        30
     Class 3     0.2903    0.7826    0.4235        46
     Class 4     0.1282    0.1064    0.1163        47
     Class 5     0.0000    0.0000    0.0000        21
     Class 6     0.2609    0.2824    0.2712        85
     Class 7     0.0000    0.0000    0.0000        28
     Class 8     0.3000    0.1277    0.1791        47
     Class 9     0.4609    0.7681    0.5761        69
    Class 10     0.1667    0.0233    0.0408        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.2951    0.4865    0.3673        37
    Class 13     1.0000    0.3214    0.4865        28
    Class 14     0.3153    0.5303    0.3955        66
    Class 15     0.2568    0.365

...


Epoch 13/15
  Train Loss: 2.8557
  Val Loss:   1.7363
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.6000    0.0732    0.1304        41
     Class 1     0.4842    0.5055    0.4946        91
     Class 2     0.3000    0.5000    0.3750        30
     Class 3     0.7381    0.6739    0.7045        46
     Class 4     0.4151    0.4681    0.4400        47
     Class 5     0.2667    0.1905    0.2222        21
     Class 6     0.4066    0.4353    0.4205        85
     Class 7     0.5714    0.1429    0.2286        28
     Class 8     0.2662    0.7872    0.3978        47
     Class 9     0.6111    0.7971    0.6918        69
    Class 10     0.2812    0.2093    0.2400        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.6667    0.6486    0.6575        37
    Class 13     0.6857    0.8571    0.7619        28
    Class 14     0.6957    0.4848    0.5714        66
    Class 15     0.7576    0.48

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 14/15
  Train Loss: 2.7610
  Val Loss:   1.7265
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.7500    0.2195    0.3396        41
     Class 1     0.6471    0.4835    0.5535        91
     Class 2     0.3077    0.2667    0.2857        30
     Class 3     0.8140    0.7609    0.7865        46
     Class 4     0.4062    0.2766    0.3291        47
     Class 5     0.3077    0.1905    0.2353        21
     Class 6     0.3727    0.4824    0.4205        85
     Class 7     0.8571    0.2143    0.3429        28
     Class 8     0.3298    0.6596    0.4397        47
     Class 9     0.7286    0.7391    0.7338        69
    Class 10     0.3125    0.4651    0.3738        43
    Class 11     0.5714    0.1538    0.2424        26
    Class 12     0.4894    0.6216    0.5476        37
    Class 13     1.0000    0.6786    0.8085        28
    Class 14     0.5625    0.2727    0.3673        66
    Class 15     0.6800    0.32

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 15/15
  Train Loss: 2.6978
  Val Loss:   1.6472
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.5000    0.4390    0.4675        41
     Class 1     0.6500    0.4286    0.5166        91
     Class 2     0.3548    0.3667    0.3607        30
     Class 3     0.7778    0.7609    0.7692        46
     Class 4     0.4340    0.4894    0.4600        47
     Class 5     0.4545    0.2381    0.3125        21
     Class 6     0.4940    0.4824    0.4881        85
     Class 7     0.3810    0.2857    0.3265        28
     Class 8     0.3768    0.5532    0.4483        47
     Class 9     0.3721    0.9275    0.5311        69
    Class 10     0.4000    0.4186    0.4091        43
    Class 11     0.5833    0.2692    0.3684        26
    Class 12     0.5294    0.4865    0.5070        37
    Class 13     0.7857    0.7857    0.7857        28
    Class 14     0.6800    0.5152    0.5862        66
    Class 15     0.6207    0.34

## Session 03

### Work 01: Build Neural Network

In [44]:
from src.networks.model03 import model03

In [48]:
print(model03(num_classes=len(classes)))

model03(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), 

### Work 02: Train model

In [50]:
def train_instance03(
    # --- Nhóm Cốt lõi ---
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    
    # --- Nhóm Cấu hình ---
    num_epochs: int,
    device: torch.device,
    
    # --- Nhóm MLOps ---
    checkpoint_dir: str,
    logger: SummaryWriter
):
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
        
    best_val_loss = float('inf')
    
    print(f"Bắt đầu training trên thiết bị: {device}")
    
    for epoch in range(num_epochs):
        
        # ==========================
        #      PHA TRAINING
        # ==========================
        model.train() # Bật chế độ train (quan trọng cho BatchNorm, Dropout)
        running_train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            # --- LOGIC CỦA RESNET (1 OUTPUT) ---
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            # --- KẾT THÚC LOGIC CỦA RESNET ---
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        
        # ==========================
        #     PHA VALIDATION
        # ==========================
        model.eval() # Bật chế độ eval
        running_val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # ResNet chỉ có 1 output ngay cả khi train hay eval
                outputs = model(inputs) 
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                _, preds = torch.max(outputs, 1)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        avg_val_loss = running_val_loss / len(val_loader)
        
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        
        # ==========================
        #     LOGGING & CHECKPOINT
        # ==========================
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {avg_val_loss:.4f}")
        
        logger.add_scalar('Loss/train', avg_train_loss, epoch)
        logger.add_scalar('Loss/validation', avg_val_loss, epoch)
        
        # In classification report (precision, recall, f1)
        try:
            class_names = val_loader.dataset.classes
        except:
            class_names = [f'Class {i}' for i in range(len(np.unique(all_labels)))]
            
        report = classification_report(
            all_labels, 
            all_preds, 
            target_names=class_names, 
            zero_division=0,
            digits=4
        )
        print("--- Báo cáo đánh giá (Precision, Recall, F1) ---")
        print(report)
        
        report_dict = classification_report(all_labels, all_preds, zero_division=0, output_dict=True)
        logger.add_scalar('F1-Score/macro_avg', report_dict['macro avg']['f1-score'], epoch)
        logger.add_scalar('Accuracy/validation', report_dict['accuracy'], epoch)

        # Lưu checkpoint
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save(model.state_dict(), save_path)
            print(f"==> Model tốt nhất đã được lưu tại: {save_path}")
            
    print("\nTraining hoàn tất.")
    logger.close()
    return None

In [51]:
# --- Các biến cấu hình ---
NUM_CLASSES = 21
NUM_EPOCHS = 15
LEARNING_RATE = 0.005 # Sửa 0.005 thành 0.001 có thể sẽ tốt hơn cho Adam
MODEL_NAME = "ResNet18"
CHECKPOINT_DIR = f"./checkpoints/{MODEL_NAME}"
LOG_DIR = f"./logs/{MODEL_NAME}_{int(time.time())}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [52]:
print(f"Đang khởi tạo các thành phần cho {MODEL_NAME}...")
instance03 = model03(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(instance03.parameters(), lr=LEARNING_RATE) 
logger = SummaryWriter(log_dir=LOG_DIR)

print(f"Model: {MODEL_NAME}, Num Classes: {NUM_CLASSES}")
print(f"Device: {device}")
print(f"Checkpoints sẽ lưu tại: {CHECKPOINT_DIR}")
print(f"Logs TensorBoard sẽ lưu tại: {LOG_DIR}")

Đang khởi tạo các thành phần cho ResNet18...
Model: ResNet18, Num Classes: 21
Device: cuda
Checkpoints sẽ lưu tại: ./checkpoints/ResNet18
Logs TensorBoard sẽ lưu tại: ./logs/ResNet18_1762085720


In [53]:
print("Bắt đầu huấn luyện...")
train_instance03(
    model=instance03,
    train_loader=train, 
    val_loader=val,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger
)

Bắt đầu huấn luyện...
Bắt đầu training trên thiết bị: cuda


c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 1/15
  Train Loss: 2.8706
  Val Loss:   2.6957
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000        41
     Class 1     0.1446    0.1319    0.1379        91
     Class 2     0.0877    0.1667    0.1149        30
     Class 3     0.0964    0.4130    0.1564        46
     Class 4     0.0000    0.0000    0.0000        47
     Class 5     0.0000    0.0000    0.0000        21
     Class 6     0.1613    0.4118    0.2318        85
     Class 7     0.0000    0.0000    0.0000        28
     Class 8     0.1429    0.0638    0.0882        47
     Class 9     0.2487    0.6957    0.3664        69
    Class 10     0.0000    0.0000    0.0000        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.3684    0.3784    0.3733        37
    Class 13     0.6957    0.5714    0.6275        28
    Class 14     0.2419    0.2273    0.2344        66
    Class 15     0.0000    0.000

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 2/15
  Train Loss: 2.6198
  Val Loss:   2.5435
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000        41
     Class 1     0.5000    0.0220    0.0421        91
     Class 2     0.0787    0.3333    0.1274        30
     Class 3     0.3214    0.3913    0.3529        46
     Class 4     0.0000    0.0000    0.0000        47
     Class 5     0.0000    0.0000    0.0000        21
     Class 6     0.1547    0.3294    0.2105        85
     Class 7     0.0000    0.0000    0.0000        28
     Class 8     0.2000    0.0213    0.0385        47
     Class 9     0.2361    0.7971    0.3642        69
    Class 10     0.5000    0.0233    0.0444        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.2821    0.2973    0.2895        37
    Class 13     0.6000    0.6429    0.6207        28
    Class 14     0.1901    0.4091    0.2596        66
    Class 15     0.2308    0.288

...


Epoch 13/15
  Train Loss: 1.8421
  Val Loss:   1.9353
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.4348    0.2439    0.3125        41
     Class 1     0.8000    0.1758    0.2883        91
     Class 2     0.2111    0.6333    0.3167        30
     Class 3     0.6757    0.5435    0.6024        46
     Class 4     0.2800    0.2979    0.2887        47
     Class 5     0.1667    0.0476    0.0741        21
     Class 6     0.3121    0.5176    0.3894        85
     Class 7     0.2500    0.0357    0.0625        28
     Class 8     0.2456    0.2979    0.2692        47
     Class 9     0.7258    0.6522    0.6870        69
    Class 10     0.3529    0.4186    0.3830        43
    Class 11     1.0000    0.1154    0.2069        26
    Class 12     0.4348    0.5405    0.4819        37
    Class 13     0.9545    0.7500    0.8400        28
    Class 14     0.7692    0.3030    0.4348        66
    Class 15     0.5778    0.50

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 14/15
  Train Loss: 1.7905
  Val Loss:   2.0893
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.8000    0.0976    0.1739        41
     Class 1     0.5781    0.4066    0.4774        91
     Class 2     0.2759    0.2667    0.2712        30
     Class 3     0.6875    0.7174    0.7021        46
     Class 4     0.3913    0.1915    0.2571        47
     Class 5     0.0909    0.0952    0.0930        21
     Class 6     0.3103    0.2118    0.2517        85
     Class 7     0.6000    0.1071    0.1818        28
     Class 8     0.3088    0.4468    0.3652        47
     Class 9     0.6935    0.6232    0.6565        69
    Class 10     0.1667    0.0233    0.0408        43
    Class 11     0.5000    0.1154    0.1875        26
    Class 12     0.4062    0.7027    0.5149        37
    Class 13     0.8333    0.7143    0.7692        28
    Class 14     0.6316    0.3636    0.4615        66
    Class 15     0.3690    0.59

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 15/15
  Train Loss: 1.7497
  Val Loss:   1.8787
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.4706    0.3902    0.4267        41
     Class 1     0.5410    0.3626    0.4342        91
     Class 2     0.3158    0.2000    0.2449        30
     Class 3     0.3727    0.8913    0.5256        46
     Class 4     0.2973    0.4681    0.3636        47
     Class 5     0.2500    0.0476    0.0800        21
     Class 6     0.5385    0.2471    0.3387        85
     Class 7     0.3333    0.1786    0.2326        28
     Class 8     0.3953    0.3617    0.3778        47
     Class 9     0.7778    0.6087    0.6829        69
    Class 10     0.3478    0.1860    0.2424        43
    Class 11     0.5000    0.0769    0.1333        26
    Class 12     0.3247    0.6757    0.4386        37
    Class 13     0.8636    0.6786    0.7600        28
    Class 14     0.3636    0.5455    0.4364        66
    Class 15     0.5600    0.53

## Session 04

### Work 01: Load Neural Network

In [54]:
from src.networks.pretrained_resnet import PretrainedResnet

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [56]:
instance04 = PretrainedResnet()
print(instance04)

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VICTUS\.cache\huggingface\hub\models--microsoft--resnet-50. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is n

PretrainedResnet(
  (resnet): ResNetModel(
    (embedder): ResNetEmbeddings(
      (embedder): ResNetConvLayer(
        (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activation): ReLU()
      )
      (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (encoder): ResNetEncoder(
      (stages): ModuleList(
        (0): ResNetStage(
          (layers): Sequential(
            (0): ResNetBottleNeckLayer(
              (shortcut): ResNetShortCut(
                (convolution): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              )
              (layer): Sequential(
                (0): ResNetConvLayer(
                  (convolution): Conv2d(64, 64, kernel

### Work 02: Finetune model

In [61]:
def setup_finetune_model(learning_rate, device):
    """
    Hàm này khởi tạo mô hình PretrainedResnet, đóng băng các lớp 
    và trả về mô hình cùng optimizer đã được cấu hình 
    chỉ để finetune lớp classifier.
    """
    
    # 1. Khởi tạo mô hình (KHÔNG truyền num_classes)
    print(f"Đang tải PretrainedResnet (ResNet-50)...")
    model = PretrainedResnet().to(device) # <--- ĐÃ SỬA
    
    # 2. Đóng băng các lớp ResNet body
    print("Đang đóng băng (freezing) các lớp 'resnet' (body)...")
    for param in model.resnet.parameters():
        param.requires_grad = False
        
    # 3. Tạo optimizer CHỈ cho các tham số của classifier
    print("Cấu hình Optimizer chỉ để train lớp 'classifier'...")
    optimizer = optim.Adam(model.classifier.parameters(), lr=learning_rate)
    
    return model, optimizer

In [59]:
def train_instance04(
    # --- Nhóm Cốt lõi ---
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    
    # --- Nhóm Cấu hình ---
    num_epochs: int,
    device: torch.device,
    
    # --- Nhóm MLOps ---
    checkpoint_dir: str,
    logger: SummaryWriter
):
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
        
    best_val_loss = float('inf')
    
    print(f"Bắt đầu training trên thiết bị: {device}")
    
    for epoch in range(num_epochs):
        
        # ==========================
        #      PHA TRAINING
        # ==========================
        model.train() # Bật chế độ train (quan trọng cho BatchNorm, Dropout)
        running_train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            # --- LOGIC CỦA RESNET (1 OUTPUT) ---
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            # --- KẾT THÚC LOGIC CỦA RESNET ---
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        
        # ==========================
        #     PHA VALIDATION
        # ==========================
        model.eval() # Bật chế độ eval
        running_val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # ResNet chỉ có 1 output ngay cả khi train hay eval
                outputs = model(inputs) 
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                _, preds = torch.max(outputs, 1)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        avg_val_loss = running_val_loss / len(val_loader)
        
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        
        # ==========================
        #     LOGGING & CHECKPOINT
        # ==========================
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {avg_val_loss:.4f}")
        
        logger.add_scalar('Loss/train', avg_train_loss, epoch)
        logger.add_scalar('Loss/validation', avg_val_loss, epoch)
        
        # In classification report (precision, recall, f1)
        try:
            class_names = val_loader.dataset.classes
        except:
            class_names = [f'Class {i}' for i in range(len(np.unique(all_labels)))]
            
        report = classification_report(
            all_labels, 
            all_preds, 
            target_names=class_names, 
            zero_division=0,
            digits=4
        )
        print("--- Báo cáo đánh giá (Precision, Recall, F1) ---")
        print(report)
        
        report_dict = classification_report(all_labels, all_preds, zero_division=0, output_dict=True)
        logger.add_scalar('F1-Score/macro_avg', report_dict['macro avg']['f1-score'], epoch)
        logger.add_scalar('Accuracy/validation', report_dict['accuracy'], epoch)

        # Lưu checkpoint
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save(model.state_dict(), save_path)
            print(f"==> Model tốt nhất đã được lưu tại: {save_path}")
            
    print("\nTraining hoàn tất.")
    logger.close()
    return None

In [64]:
NUM_CLASSES = 21
NUM_EPOCHS = 15
LEARNING_RATE = 0.001
MODEL_NAME = "ResNet50_Finetune"
CHECKPOINT_DIR = f"./checkpoints/{MODEL_NAME}"
LOG_DIR = f"./logs/{MODEL_NAME}_{int(time.time())}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [69]:
print(f"Đang khởi tạo các thành phần cho {MODEL_NAME}...")

instance04, optimizer = setup_finetune_model(
    learning_rate=LEARNING_RATE,
    device=device
)

criterion = nn.CrossEntropyLoss()
logger = SummaryWriter(log_dir=LOG_DIR)

Đang khởi tạo các thành phần cho ResNet50_Finetune...
Đang tải PretrainedResnet (ResNet-50)...
Đang đóng băng (freezing) các lớp 'resnet' (body)...
Cấu hình Optimizer chỉ để train lớp 'classifier'...


In [ ]:
print("\nBắt đầu huấn luyện (chế độ Finetuning)...")
train_instance03(
    model=instance04,
    train_loader=train, 
    val_loader=val,     
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger
)


Bắt đầu huấn luyện (chế độ Finetuning)...
Bắt đầu training trên thiết bị: cuda


c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 1/15
  Train Loss: 2.4504
  Val Loss:   2.0462
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.8947    0.4146    0.5667        41
     Class 1     0.3228    0.5604    0.4096        91
     Class 2     0.4103    0.5333    0.4638        30
     Class 3     0.5714    0.8696    0.6897        46
     Class 4     0.6512    0.5957    0.6222        47
     Class 5     0.7778    0.3333    0.4667        21
     Class 6     0.4865    0.4235    0.4528        85
     Class 7     1.0000    0.3571    0.5263        28
     Class 8     0.5278    0.4043    0.4578        47
     Class 9     0.3925    0.6087    0.4773        69
    Class 10     1.0000    0.0698    0.1304        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.5349    0.6216    0.5750        37
    Class 13     0.6154    0.5714    0.5926        28
    Class 14     0.4889    0.6667    0.5641        66
    Class 15     0.7179    0.538

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 2/15
  Train Loss: 1.8595
  Val Loss:   1.7013
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.5610    0.5610    0.5610        41
     Class 1     0.5088    0.6374    0.5659        91
     Class 2     0.4074    0.7333    0.5238        30
     Class 3     0.7069    0.8913    0.7885        46
     Class 4     0.5814    0.5319    0.5556        47
     Class 5     0.5833    0.6667    0.6222        21
     Class 6     0.4742    0.5412    0.5055        85
     Class 7     0.9412    0.5714    0.7111        28
     Class 8     0.5938    0.4043    0.4810        47
     Class 9     0.5143    0.5217    0.5180        69
    Class 10     0.6364    0.3256    0.4308        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.5500    0.5946    0.5714        37
    Class 13     0.8077    0.7500    0.7778        28
    Class 14     0.6620    0.7121    0.6861        66
    Class 15     0.6170    0.557

...

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 13/15
  Train Loss: 1.0588
  Val Loss:   1.1308
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.7742    0.5854    0.6667        41
     Class 1     0.6145    0.5604    0.5862        91
     Class 2     0.4468    0.7000    0.5455        30
     Class 3     0.9333    0.9130    0.9231        46
     Class 4     0.8049    0.7021    0.7500        47
     Class 5     0.8889    0.7619    0.8205        21
     Class 6     0.5408    0.6235    0.5792        85
     Class 7     1.0000    0.8571    0.9231        28
     Class 8     0.7667    0.4894    0.5974        47
     Class 9     0.5851    0.7971    0.6748        69
    Class 10     0.5610    0.5349    0.5476        43
    Class 11     0.3077    0.1538    0.2051        26
    Class 12     0.6829    0.7568    0.7179        37
    Class 13     0.8000    0.7143    0.7547        28
    Class 14     0.6364    0.7424    0.6853        66
    Class 15     0.7358    0.75

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 14/15
  Train Loss: 1.0531
  Val Loss:   1.0673
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.7576    0.6098    0.6757        41
     Class 1     0.5546    0.7253    0.6286        91
     Class 2     0.5263    0.6667    0.5882        30
     Class 3     0.7627    0.9783    0.8571        46
     Class 4     0.7778    0.7447    0.7609        47
     Class 5     0.6667    0.6667    0.6667        21
     Class 6     0.5234    0.6588    0.5833        85
     Class 7     1.0000    0.7500    0.8571        28
     Class 8     0.7105    0.5745    0.6353        47
     Class 9     0.7465    0.7681    0.7571        69
    Class 10     0.6585    0.6279    0.6429        43
    Class 11     0.5000    0.2308    0.3158        26
    Class 12     0.7333    0.8919    0.8049        37
    Class 13     0.8889    0.8571    0.8727        28
    Class 14     0.8431    0.6515    0.7350        66
    Class 15     0.7292    0.67

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
